# Lab 07 — Delta Live Tables (DLT) Pipeline

## Declarative ETL: Define WHAT you want, not HOW to build it

---

### What is DLT?

In Labs 04-06, you built the Medallion pipeline manually: read → transform → write for each layer.  
**Delta Live Tables** lets you declare each table as a function, and DLT handles:
- Execution order (dependency graph)
- Retries and error recovery
- Data quality monitoring (expectations)
- Schema evolution

### How to Run This Notebook

> **Do NOT click "Run All"** — DLT notebooks are executed by the DLT engine.
>
> 1. Go to **Workflows → Delta Live Tables** in the left sidebar
> 2. Click **Create Pipeline**
> 3. Pipeline name: `claims_medallion_pipeline`
> 4. Source: select this notebook
> 5. Target Schema: create or select a schema (e.g., `dlt_lab`)
> 6. Compute: **Serverless** or your cluster
> 7. Click **Start**
>
> After the pipeline runs, click on each table node in the DAG to see:
> - **Expectations tab**: data quality pass/fail rates
> - **Schema tab**: column types
> - **Details tab**: record counts, duration

### Pipeline Architecture

```
claims_raw (Hive table from Lab 03)
    │
    ▼
┌──────────────────┐
│  claims_bronze   │  Raw data + ingestion timestamp
└──────┬───────────┘
       │
       ▼
┌──────────────────┐
│  claims_silver   │  Cleaned: nulls filtered, amounts validated
│  4 expectations  │  2x WARN (allow) + 2x DROP
└──────┬───────────┘
       │
       ├──────────────────┐──────────────────┐
       ▼                  ▼                  ▼
┌──────────────┐  ┌──────────────┐  ┌──────────────┐
│ gold_summary │  │  gold_top5   │  │claims_strict │
│ groupBy agg  │  │ window rank  │  │ expect_fail  │
└──────────────┘  └──────────────┘  └──────────────┘
```

---
## Step 1: Bronze Layer — Raw Ingestion

The Bronze layer captures raw data exactly as-is, plus an `_ingested_at` timestamp.  
- `@dlt.table` declares a table managed by DLT  
- `dlt.read()` reads from another DLT table (used in Silver/Gold)  
- Here we use `spark.sql()` to read from the Hive table created in Lab 03

In [ ]:
import dlt
from pyspark.sql.functions import current_timestamp, col

# Bronze: ingest raw claims from the Hive table
# @dlt.table tells DLT to manage this as a Delta table
# The function returns a DataFrame — DLT handles the write

@dlt.table(
    name="claims_bronze",
    comment="Raw claims data — no transformations, just an ingestion timestamp"
)
def claims_bronze():
    return (
        spark.sql("SELECT * FROM training_lab.claims_raw")
        .withColumn("_ingested_at", current_timestamp())
    )

---
## Step 2: Silver Layer — Data Quality with Expectations

DLT **expectations** define data quality rules directly on the table.  
Three modes — this is what makes DLT special:

| Decorator | Action | What happens on violation |
|-----------|--------|-------------------------|
| `@dlt.expect("name", "condition")` | **WARN** | Row is kept, violation is logged (visible in Data Quality tab) |
| `@dlt.expect_or_drop("name", "condition")` | **DROP** | Row is silently removed |
| `@dlt.expect_or_fail("name", "condition")` | **FAIL** | Entire pipeline stops |

After the pipeline runs, click on `claims_silver` in the DAG → **Expectations** tab to see pass/fail rates.

In [ ]:
# Silver: clean and validate claims
#
# @dlt.expect = WARN mode:
#   - valid_claim_id: logs if any claim_id is NULL (but keeps the row)
#   - reasonable_amount: logs if claim_amount >= 400,000 (flags high-value claims)
#
# @dlt.expect_or_drop = DROP mode:
#   - valid_amount: removes rows where claim_amount <= 0
#   - valid_status: removes rows with unexpected status values
#
# dlt.read("claims_bronze") reads from the Bronze table defined above
# DLT knows the dependency and runs Bronze first automatically

@dlt.table(
    name="claims_silver",
    comment="Cleaned claims — nulls removed, amounts validated, types checked"
)
@dlt.expect("valid_claim_id", "claim_id IS NOT NULL")
@dlt.expect("reasonable_amount", "claim_amount < 400000")
@dlt.expect_or_drop("valid_amount", "claim_amount > 0")
@dlt.expect_or_drop("valid_status", "claim_status IN ('OPEN', 'CLOSED', 'PENDING', 'UNDER_REVIEW')")
def claims_silver():
    return (
        dlt.read("claims_bronze")
        .filter(col("claim_id").isNotNull())
        .filter(col("claim_amount").isNotNull())
        .select(
            "claim_id",
            "policy_id",
            "axa_entity_id",
            col("claim_date").cast("timestamp").alias("claim_date"),
            "claim_amount",
            "claim_type",
            "claim_status",
            "adjuster_id",
            "_ingested_at"
        )
    )

---
## Step 3: Gold Layer — Business Aggregations

Gold tables produce business-ready results. DLT automatically chains them after Silver.

In [ ]:
from pyspark.sql import functions as F

# Gold: claims summary per entity and claim type
# This is the same groupBy/agg you did in Lab 06, but declared as a DLT table
# DLT handles: execution order, retries, schema management

@dlt.table(
    name="claims_gold_summary",
    comment="Business aggregate — claims count, total, and average per entity and type"
)
def claims_gold_summary():
    return (
        dlt.read("claims_silver")
        .groupBy("axa_entity_id", "claim_type")
        .agg(
            F.count("claim_id").alias("claim_count"),
            F.sum("claim_amount").alias("total_amount"),
            F.round(F.avg("claim_amount"), 2).alias("avg_amount")
        )
    )

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

# Gold: top 5 claims per entity by amount
# Uses a window function (dense_rank) — same concept as Lab 02 Exercise 6
# Useful for risk monitoring: which entities have the largest claims?

@dlt.table(
    name="claims_gold_top5",
    comment="Top 5 highest claims per entity — useful for risk monitoring"
)
def claims_gold_top5():
    w = Window.partitionBy("axa_entity_id").orderBy(col("claim_amount").desc())
    return (
        dlt.read("claims_silver")
        .withColumn("rank", dense_rank().over(w))
        .filter(col("rank") <= 5)
    )

---
## Step 4: Critical Expectations — FAIL Mode

`@dlt.expect_or_fail` is the strictest mode:  
if **any single row** violates the rule, the **entire pipeline stops**.  
Use this for critical business rules where bad data must never propagate downstream.

In [ ]:
# Strict validation: pipeline FAILS if any claim_date or entity is null
# Since claims_silver already filters nulls, this should pass
# But if upstream data changes and nulls slip through, the pipeline stops immediately

@dlt.table(
    name="claims_strict",
    comment="Strict validation — pipeline fails if any claim_date or entity is null"
)
@dlt.expect_or_fail("date_not_null", "claim_date IS NOT NULL")
@dlt.expect_or_fail("entity_not_null", "axa_entity_id IS NOT NULL")
def claims_strict():
    return dlt.read("claims_silver")

---
## After the Pipeline Runs

### What to Check

1. **DAG Graph**: see the dependency chain Bronze → Silver → Gold
2. **Click on `claims_silver`** → **Expectations** tab:
   - `valid_claim_id`: ALLOW — should be 0% fail
   - `reasonable_amount`: ALLOW — ~20% fail (claims > 400K are flagged but kept)
   - `valid_amount`: DROP — removes any negative amounts
   - `valid_status`: DROP — removes invalid statuses
3. **Click on `claims_strict`** → Expectations tab: both should be 0% fail
4. **Query the tables** in a new notebook:
   ```python
   display(spark.table("dlt_lab.claims_gold_summary"))
   display(spark.table("dlt_lab.claims_gold_top5"))
   ```

### Key Takeaways

| Concept | Traditional ETL | DLT |
|---------|----------------|-----|
| Execution order | You manage it | Automatic from `dlt.read()` dependencies |
| Data quality | Custom assert/filter code | `@dlt.expect` decorators with 3 modes |
| Monitoring | Build your own dashboards | Built-in Data Quality tab |
| Schema changes | Pipeline breaks | Handled automatically |
| Retries | Custom error handling | Automatic with checkpointing |